# NETRA Phase 2 — DistilBERT Tier-1 Training (LOCAL GPU EDITION)

**Project:** NETRA — The AI Eye Against Phishing  
**Goal:** Fine-tune DistilBERT locally on NVIDIA GPU (RTX 3050 / 40-series / 50-series) with zero cloud dependencies.

---

### Instructions:
1. Open this file directly in **VS Code** or **JupyterLab**.
2. Select your local Python environment (with PyTorch + CUDA enabled).
3. Click **Run All** (or run cells one-by-one).


## Section 1: Check Local GPU & Environment

Verifies your NVIDIA CUDA GPU, VRAM, and PyTorch configuration.


In [ ]:
import torch

print('PyTorch Version :', torch.__version__)
print('CUDA Available  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device Name :', torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print(f'Total VRAM      : {props.total_memory / 1e9:.2f} GB')
else:
    print('WARNING: CUDA is not available. Please install PyTorch with CUDA support if you have an NVIDIA GPU.')


## Section 2: Local Project Paths Setup

Automatically discovers your local NETRA directory paths.


In [ ]:
from pathlib import Path
import sys, os, json

# Locate repo root directory automatically
CURRENT_DIR = Path('.').resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == 'notebooks' else CURRENT_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_CSV      = PROJECT_ROOT / 'data' / 'processed' / 'unified.csv'
RAW_DIR       = PROJECT_ROOT / 'data' / 'raw'
MODELS_DIR    = PROJECT_ROOT / 'ml' / 'models'
EVAL_DIR      = PROJECT_ROOT / 'ml' / 'evaluation'
NOTEBOOKS_DIR = PROJECT_ROOT / 'notebooks'

for d in [RAW_DIR, MODELS_DIR, EVAL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'NETRA Root  : {PROJECT_ROOT}')
print(f'Data dir    : {RAW_DIR}')
print(f'Models dir  : {MODELS_DIR}')


## Section 3: Download Raw Datasets

Downloads all required training data:  
- **SpamAssassin** public corpus (legitimate + spam emails)
- **Nazario phishing mbox** (real phishing emails)
- **PhishTank CSV** (phishing URLs as email bodies)
- **OpenPhish feed** (phishing URLs)
- **Enron CSV** (legitimate emails from Kaggle, if Kaggle credentials provided)

> If Enron CSV is unavailable, the pipeline will still produce a valid dataset
> using SpamAssassin + Nazario + PhishTank. Training will still work.

**Each download is skipped if the file already exists** (safe to re-run).

In [ ]:
import urllib.request, tarfile, zipfile, os
RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

def download_if_missing(url, dest):
    """Download url to dest path, skip if already exists."""
    dest = Path(dest)
    if dest.exists():
        print(f"  Already exists: {dest.name}")
        return
    print(f"  Downloading {dest.name}...")
    urllib.request.urlretrieve(url, str(dest))
    print(f"  Done: {dest.name} ({dest.stat().st_size // 1024} KB)")

# ─── SpamAssassin public corpus ───────────────────────────────
print("[1/4] SpamAssassin corpus")
SA_BASE = 'https://spamassassin.apache.org/old/publiccorpus'
SA_ARCHIVES = [
    ("20030228_easy_ham.tar.bz2",   "easy_ham"),
    ("20030228_hard_ham.tar.bz2",   "hard_ham"),
    ("20030228_spam.tar.bz2",       "spam"),
    ("20050311_spam_2.tar.bz2",     "spam_2"),
]
for archive_name, folder_name in SA_ARCHIVES:
    dest_folder = RAW_DIR / folder_name
    if dest_folder.exists() and any(dest_folder.iterdir()):
        print(f"  Already exists: {folder_name}/")
        continue
    archive_path = RAW_DIR / archive_name
    download_if_missing(f'{SA_BASE}/{archive_name}', archive_path)
    print(f"  Extracting {archive_name}...")
    with tarfile.open(str(archive_path), 'r:bz2') as tar:
        tar.extractall(str(RAW_DIR))
    archive_path.unlink()  # remove archive to save space

# ─── Nazario phishing mbox ────────────────────────────────────
print("[2/4] Nazario phishing corpus")
nazario_path = RAW_DIR / 'phishing3.mbox'
if not nazario_path.exists():
    try:
        # Jose Nazario's active public archive on monkey.org (~20MB mbox)
        NAZARIO_URL = 'https://monkey.org/~jose/phishing/phishing3.mbox'
        download_if_missing(NAZARIO_URL, nazario_path)
    except Exception as e:
        print(f"  [!] Nazario download failed ({e}) — continuing with SpamAssassin + URL feeds.")
        # Create an empty mbox stub so pipeline doesn't break
        nazario_path.touch()

# ─── PhishTank CSV ────────────────────────────────────────────
print("[3/4] PhishTank phishing URLs")
# Download from PhishTank (no auth needed for verified_online)
PHISHTANK_URL = 'https://data.phishtank.com/data/online-valid.csv'
phishtank_path = RAW_DIR / 'phishtank_online_valid.csv'
if not phishtank_path.exists():
    try:
        download_if_missing(PHISHTANK_URL, phishtank_path)
    except Exception as e:
        print(f"  PhishTank download failed ({e}) — creating minimal stub")
        # Create a minimal stub so the pipeline doesn't crash
        phishtank_path.write_text("phish_id,url,phish_detail_url,submission_time,verified,verification_time,online,target\n"
                                  "1,http://phishing.example.com/login,,2024-01-01,yes,2024-01-01,yes,PayPal\n")

# ─── OpenPhish feed ───────────────────────────────────────────
print("[4/4] OpenPhish URL feed")
OPENPHISH_URL = 'https://openphish.com/feed.txt'
openphish_path = RAW_DIR / 'openphish_feed.txt'
if not openphish_path.exists():
    try:
        download_if_missing(OPENPHISH_URL, openphish_path)
    except Exception as e:
        print(f"  OpenPhish download failed ({e}) — creating minimal stub")
        openphish_path.write_text(
            "http://malicious-phishing-example.tk/steal-credentials\n"
            "http://fake-paypal-login.ml/account\n"
        )

print("\nAll datasets ready.")


### Optional: Enron Email Dataset from Kaggle

Enron adds ~517,000 legitimate emails and greatly improves model accuracy.

**To enable:**
1. Go to [kaggle.com/account](https://www.kaggle.com/account) → API → Create New Token
2. Download `kaggle.json` and upload it to this Colab session
3. Run the cell below

> **Skip this cell** if you don't have Kaggle credentials — training will still work.

In [ ]:
# OPTIONAL — Enron dataset from Kaggle
# Uncomment and run only if you have kaggle.json uploaded

# import shutil
# !mkdir -p ~/.kaggle
# !cp /content/kaggle.json ~/.kaggle/kaggle.json
# !chmod 600 ~/.kaggle/kaggle.json
# !pip install kaggle --quiet
# !kaggle datasets download -d wcukierski/enron-email-dataset -p /tmp/enron --unzip
# shutil.move("/tmp/enron/emails.csv", str(RAW_DIR / "emails.csv"))
# print("Enron CSV downloaded:", (RAW_DIR / "emails.csv").stat().st_size // 1024 // 1024, "MB")

print("Enron cell skipped (optional). Add Kaggle credentials above to enable.")

## Section 4: Build unified.csv (Data Pipeline)

Runs `ml/data_pipeline.py` to parse all raw datasets, extract email features,
deduplicate, assign labels (0=legitimate, 1=phishing), and split into train/val/test.  

Output: `data/processed/unified.csv` (~300MB–1.5GB depending on whether Enron was downloaded)

> **This cell is skipped if `unified.csv` already exists** — safe to re-run.

In [ ]:
import subprocess
import sys

if DATA_CSV.exists():
    print(f"unified.csv already exists ({DATA_CSV.stat().st_size // 1024 // 1024} MB) — skipping pipeline.")
else:
    print("Running data pipeline (this may take 2–5 minutes)...")
    pipeline_script = PROJECT_ROOT / "ml" / "data_pipeline.py"
    result = subprocess.run(
        [sys.executable, str(pipeline_script)],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print("PIPELINE STDERR:\n", result.stderr[-3000:])
        raise RuntimeError("data_pipeline.py failed! See stderr above.")
    print(result.stdout[-2000:])
    print(f"unified.csv created: {DATA_CSV.stat().st_size // 1024 // 1024} MB")


## Section 5: Why DistilBERT?

### The Problem with TF-IDF + Random Forest (Phase 1)

| Limitation | Impact |
|---|---|
| TF-IDF treats each word independently | Cannot understand context or intent |
| Typosquat `paypa1` = unknown token | Misses domain spoofing completely |
| Short phishing emails = sparse vectors | Very low feature signal |
| 90:10 class imbalance | Biased toward predicting "Legitimate" |

### How DistilBERT Fixes This

```
[Email Body Text]
       |
DistilBERT (6 transformer layers — pretrained on 3.3B words)
       |                    [SPF/DKIM/DMARC signals (10 dims)]
[CLS] embedding (768d)           |
       |               Linear(10→32) → ReLU
       |_______________|  
  Concatenate [800 dims]
       |
  Linear(800→256) → ReLU → Dropout(0.3) → Linear(256→2)
       |
 [LEGITIMATE=0 / PHISHING=1]
```

**SUSPICIOUS** is inference-time only: if `max(softmax_prob) < 0.70` → classify as `SUSPICIOUS`

## Section 6: Load Data & Verify Class Distribution

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

HEADER_FEATURE_NAMES = [
    "spf_pass", "spf_fail", "spf_none",
    "dkim_pass", "dkim_fail", "dkim_none",
    "dmarc_pass", "dmarc_fail", "dmarc_none",
    "sender_domain_match",
]

print(f"Loading: {DATA_CSV}")
df = pd.read_csv(DATA_CSV, dtype=str)
df = df[df["split"].isin(["train", "val"])].reset_index(drop=True)
df["label"] = df["label"].astype(int)

# Gracefully fill missing header feature columns with 0
for col in HEADER_FEATURE_NAMES:
    if col not in df.columns:
        df[col] = 0
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(float)

counts = df["label"].value_counts().sort_index()
print(f"Total : {len(df):,}")
print(f"Legit : {counts.get(0, 0):,}")
print(f"Phish : {counts.get(1, 0):,}")
print(f"Splits: {df["split"].value_counts().to_dict()}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(["LEGITIMATE", "PHISHING"], [counts.get(0,0), counts.get(1,0)],
       color=["#2ecc71", "#e74c3c"], edgecolor="black")
for i, v in enumerate([counts.get(0,0), counts.get(1,0)]):
    ax.text(i, v + 100, f"{v:,}", ha="center", fontweight="bold")
ax.set_title("Dataset Class Distribution", fontweight="bold")
plt.tight_layout()
plt.show()

if counts.get(1, 0) < 100:
    raise ValueError("Too few phishing samples! Check data_pipeline.py ran correctly.")
print("Data loaded successfully.")

## Section 7: Model Architecture & Tokenization

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, DistilBertModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
if device.type == 'cuda':
    props = torch.cuda.get_device_properties(0)
    print(f"GPU    : {props.name}")
    print(f"VRAM   : {props.total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected! Training will be very slow on CPU.")
    print("Go to Runtime -> Change runtime type -> T4 GPU")

MAX_LENGTH           = 256   # 95th pct of email body < 200 tokens; 256 is safe
BATCH_SIZE           = 32    # fits T4 16GB VRAM at max_length=256
EPOCHS               = 3     # standard for BERT fine-tuning (more = overfitting)
LR                   = 2e-5  # AdamW optimal LR for BERT
WARMUP_STEPS         = 100
CONFIDENCE_THRESHOLD = 0.70  # below this -> SUSPICIOUS at inference time
print(f"Config : max_length={MAX_LENGTH}, batch_size={BATCH_SIZE}, epochs={EPOCHS}, lr={LR}")

In [ ]:
class PhishingEmailDataset(Dataset):
    """
    Memory-efficient PyTorch Dataset:
    Tokenizes on-the-fly in workers instead of pre-tokenizing all tens of thousands
    of emails at once into Colab system RAM.
    """
    def __init__(self, texts, header_feats, labels, tokenizer, max_length=256):
        self.texts        = texts
        self.header_feats = torch.tensor(header_feats, dtype=torch.float32)
        self.labels       = torch.tensor(labels,       dtype=torch.long)
        self.tokenizer    = tokenizer
        self.max_length   = max_length

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        enc = self.tokenizer(
            str(self.texts[i]),
            max_length=self.max_length,
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        )
        return {
            "input_ids":       enc["input_ids"].squeeze(0),
            "attention_mask":  enc["attention_mask"].squeeze(0),
            "header_features": self.header_feats[i],
            "labels":          self.labels[i],
        }


class PhishingClassifier(nn.Module):
    """
    DistilBERT + header feature fusion.
    CLS embedding (768d) fused with projected header signals (32d).
    """
    def __init__(self):
        super().__init__()
        self.distilbert  = DistilBertModel.from_pretrained("distilbert-base-uncased")
        self.header_proj = nn.Linear(10, 32)
        self.classifier  = nn.Sequential(
            nn.Linear(768 + 32, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 2),  # 0=LEGITIMATE, 1=PHISHING
        )

    def forward(self, input_ids, attention_mask, header_features):
        cls = self.distilbert(
            input_ids=input_ids, attention_mask=attention_mask
        ).last_hidden_state[:, 0, :]           # [CLS] token: shape (B, 768)
        hdr = torch.relu(self.header_proj(header_features))  # shape (B, 32)
        return self.classifier(torch.cat([cls, hdr], dim=1)) # shape (B, 2)

print("Memory-efficient Dataset and Model classes defined.")


In [ ]:
import gc

print("Loading DistilBERT tokenizer...")
tokenizer    = AutoTokenizer.from_pretrained("distilbert-base-uncased")
body_texts   = df["body_text"].fillna("").astype(str).tolist()
splits_list  = df["split"].tolist()
labels_list  = df["label"].tolist()
header_feats = df[HEADER_FEATURE_NAMES].values.tolist()

# Release dataframe memory to keep Colab RAM low and healthy
del df
gc.collect()

train_idx = [i for i, s in enumerate(splits_list) if s == 'train']
val_idx   = [i for i, s in enumerate(splits_list) if s == 'val']

train_texts  = [body_texts[i] for i in train_idx]
train_hdrs   = [header_feats[i] for i in train_idx]
train_labels = [labels_list[i] for i in train_idx]

val_texts    = [body_texts[i] for i in val_idx]
val_hdrs     = [header_feats[i] for i in val_idx]
val_labels   = [labels_list[i] for i in val_idx]

# Clean up raw lists from memory
del body_texts, splits_list, labels_list, header_feats
gc.collect()

train_ds = PhishingEmailDataset(train_texts, train_hdrs, train_labels, tokenizer, max_length=MAX_LENGTH)
val_ds   = PhishingEmailDataset(val_texts,   val_hdrs,   val_labels,   tokenizer, max_length=MAX_LENGTH)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train samples : {len(train_ds):,} (Legit: {train_labels.count(0):,} | Phishing: {train_labels.count(1):,})")
print(f"Val samples   : {len(val_ds):,}")
print("Ready for training without RAM overload!")


## Section 8: Training

Fine-tunes DistilBERT for 3 epochs.  
- **Weighted cross-entropy loss** handles class imbalance (no SMOTE needed)
- **Gradient clipping** (`max_norm=1.0`) prevents exploding gradients
- **Linear warmup scheduler** avoids large early gradient steps
- **Best model checkpointing** saves the epoch with highest val F1

> Expected time: **~20–25 minutes** on Colab T4 GPU

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
from transformers import get_linear_schedule_with_warmup

# Weighted loss — gives phishing examples higher weight to counter class imbalance
cw = compute_class_weight('balanced', classes=np.array([0, 1]), y=train_labels)
criterion = nn.CrossEntropyLoss(weight=torch.tensor(cw, dtype=torch.float32).to(device))

model     = PhishingClassifier().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=WARMUP_STEPS, num_training_steps=total_steps
)

print(f"Model parameters : {sum(p.numel() for p in model.parameters()):,}")
print(f"Class weights    : Legit={cw[0]:.3f} | Phishing={cw[1]:.3f}")
print(f"Total train steps: {total_steps:,}")

In [ ]:
def evaluate(model, loader):
    """Run evaluation and return metrics dict."""
    from sklearn.metrics import (
        accuracy_score, precision_score, recall_score, f1_score, average_precision_score
    )
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for b in loader:
            logits = model(
                b['input_ids'].to(device),
                b['attention_mask'].to(device),
                b['header_features'].to(device),
            )
            all_probs.extend(torch.softmax(logits, 1)[:, 1].cpu().numpy())
            all_preds.extend(logits.argmax(1).cpu().numpy())
            all_labels.extend(b['labels'].numpy())
    preds  = np.array(all_preds)
    labels = np.array(all_labels)
    probs  = np.array(all_probs)
    fp = np.sum((preds == 1) & (labels == 0))
    tn = np.sum((preds == 0) & (labels == 0))
    return {
        'accuracy':  round(float(accuracy_score(labels, preds)), 4),
        'precision': round(float(precision_score(labels, preds, zero_division=0)), 4),
        'recall':    round(float(recall_score(labels, preds, zero_division=0)), 4),
        'f1':        round(float(f1_score(labels, preds, zero_division=0)), 4),
        'fpr':       round(float(fp / (fp + tn + 1e-9)), 4),
        'pr_auc':    round(float(average_precision_score(labels, probs)), 4),
    }
print("evaluate() defined.")

In [ ]:
try:
    from tqdm.auto import tqdm
except ImportError:
    from tqdm import tqdm

best_f1   = 0.0
best_path = MODELS_DIR / 'distilbert_tier1.pt'
history   = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss, n_batches = 0.0, 0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}', ncols=90)

    for b in pbar:
        ids  = b['input_ids'].to(device)
        mask = b['attention_mask'].to(device)
        hdrs = b['header_features'].to(device)
        lbls = b['labels'].to(device)

        optimizer.zero_grad()
        loss = criterion(model(ids, mask, hdrs), lbls)
        loss.backward()
        # Gradient clipping prevents exploding gradients during fine-tuning
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        n_batches  += 1
        pbar.set_postfix({'loss': f'{total_loss/n_batches:.4f}'})

    m = evaluate(model, val_loader)
    history.append({'epoch': epoch, 'loss': round(total_loss/n_batches, 4), **m})

    print(
        f'Epoch {epoch} | Loss={total_loss/n_batches:.4f} | '
        f'Recall={m["recall"]:.4f} F1={m["f1"]:.4f} '
        f'FPR={m["fpr"]:.4f} PR-AUC={m["pr_auc"]:.4f}'
    )

    # Save best checkpoint based on validation F1
    if m['f1'] > best_f1:
        best_f1 = m['f1']
        torch.save(model.state_dict(), best_path)
        print(f'  >> Best model checkpoint saved (F1={best_f1:.4f})')

print(f"\nTraining complete. Best val F1 = {best_f1:.4f}")


## Section 9: Results & Evaluation Plots

Load the best checkpoint and evaluate with confusion matrix + PR curve.

In [ ]:
from sklearn.metrics import (
    classification_report, confusion_matrix as sk_cm,
    precision_recall_curve, average_precision_score
)
import seaborn as sns

# Load best checkpoint
model.load_state_dict(torch.load(best_path, map_location=device))
final = evaluate(model, val_loader)

print('=' * 55)
print('Final Validation Metrics — DistilBERT Tier-1')
print('=' * 55)
for k, v in final.items():
    print(f'  {k:12}: {v:.4f}')
print()
print('Targets:')
print(f'  Phishing Recall >= 0.90 : {"ACHIEVED" if final["recall"] >= 0.90 else "NOT MET"}')
print(f'  FPR <= 0.05             : {"ACHIEVED" if final["fpr"] <= 0.05 else "NOT MET"}')

In [ ]:
# Gather predictions for plots
model.eval()
p_all, l_all, pr_all = [], [], []
with torch.no_grad():
    for b in val_loader:
        logits = model(
            b['input_ids'].to(device),
            b['attention_mask'].to(device),
            b['header_features'].to(device),
        )
        pr_all.extend(torch.softmax(logits, 1)[:, 1].cpu().numpy())
        p_all.extend(logits.argmax(1).cpu().numpy())
        l_all.extend(b['labels'].numpy())

l_all = np.array(l_all)
p_all = np.array(p_all)
pr_all = np.array(pr_all)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('NETRA DistilBERT — Validation Evaluation', fontsize=14, fontweight='bold')

# Confusion matrix
cm = sk_cm(l_all, p_all, labels=[0, 1])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['LEGITIMATE', 'PHISHING'],
            yticklabels=['LEGITIMATE', 'PHISHING'])
axes[0].set_title('Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')

# Precision-Recall curve
prec, rec, _ = precision_recall_curve(l_all, pr_all)
auc_val = average_precision_score(l_all, pr_all)
axes[1].plot(rec, prec, color='#e74c3c', lw=2, label=f'PR-AUC = {auc_val:.4f}')
axes[1].fill_between(rec, prec, alpha=0.1, color='#e74c3c')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend()

plt.tight_layout()
plot_path = EVAL_DIR / 'distilbert_evaluation.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()

print(classification_report(l_all, p_all, target_names=['LEGITIMATE', 'PHISHING']))

## Section 10: Verify Saved Artifacts

Artifacts are automatically saved directly into your local `ml/models/` folder!
1. `distilbert_tier1.pt` (~250 MB)
2. `distilbert_tokenizer/` directory
3. `distilbert_config.json`


In [ ]:
# Save tokenizer to Drive
tokenizer_dir = MODELS_DIR / 'distilbert_tokenizer'
tokenizer.save_pretrained(str(tokenizer_dir))
print('Tokenizer saved:', tokenizer_dir)

# Save inference config
db_config = {
    'model':                'distilbert-base-uncased',
    'max_length':           MAX_LENGTH,
    'confidence_threshold': CONFIDENCE_THRESHOLD,
    'header_features':      HEADER_FEATURE_NAMES,
    'best_val_f1':          final['f1'],
    'best_val_recall':      final['recall'],
    'best_val_fpr':         final['fpr'],
    'training_history':     history,
}
config_path = MODELS_DIR / 'distilbert_config.json'
with open(config_path, 'w') as f:
    json.dump(db_config, f, indent=2)
print('Config saved:', config_path)

print()
print('=' * 55)
print('All artifacts on Google Drive:')
print(f'  Model  : {best_path} ({best_path.stat().st_size // 1024 // 1024} MB)')
print(f'  Tokenizer: {tokenizer_dir}/')
print(f'  Config : {config_path}')
print('=' * 55)

In [ ]:
print('=' * 60)
print('TRAINING & EXPORT COMPLETE!')
print('=' * 60)
print(f'Model weights  : {best_path} ({best_path.stat().st_size / 1e6:.1f} MB)')
print(f'Tokenizer dir  : {tokenizer_dir}')
print(f'Model config   : {config_path}')
print('\nYou can now start the NETRA API locally by running:')
print('  python -m uvicorn api.main:app --reload --port 8000')
